## Transformers with toy corpus

In [1]:
# run on python 11
import math
import random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# Device and seeds
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Corpus for this notebook
corpus = [
    "the cat sat on the mat",
    "the dog lay on the rug",
    "dogs and cats are friends",
    "the cat chased the mouse",
    "the dog chased the cat",
    "cats and dogs can be friends",
    "dogs are friends",
    "the dog sat on the mat"
]


## 4.2 – Data: vocabulary, encoding and reversed targets

In this part we prepare the data for the Transformer, some parts are similar to the embeddings and seq2seq notebook:

1. **Build a vocabulary** from the corpus:
   - Extract all **unique tokens** from all sentences.
   - Add three special tokens:
     - `<pad>`  (padding)
     - `<bos>`  (beginning of sentence)
     - `<eos>`  (end of sentence)
   - Create:
     - `stoi`: dict mapping token → index
     - `itos`: list mapping index → token

2. **Encode sentences**:
   - A function `encode_sentence(sentence: str) -> List[int]`:
     - Tokenize the sentence by splitting on spaces.
     - Map each token to its index via `stoi`.
     - Wrap with `<bos>` at the beginning and `<eos>` at the end.

3. **Create reversed targets**:
   - A function `encode_reversed(sentence: str) -> List[int]`:
     - Tokenize the sentence.
     - Reverse the token order.
     - Map to indices.
     - Wrap with `<bos>` and `<eos>` as well.
   - Example:
     - Input: `"the cat sat on the mat"`
     - Source ids: `[BOS, "the", "cat", "sat", "on", "the", "mat", EOS]`
     - Target ids: `[BOS, "mat", "the", "on", "sat", "cat", "the", EOS]`

4. **Dataset and DataLoader**:
   - Implement a `ReversalDataset` class that:
     - Receives the `corpus`.
     - Stores `(src_ids, tgt_ids)` for each sentence.
   - Implement a `collate_fn` for the `DataLoader`:
     - Given a list of `(src_ids, tgt_ids)` pairs, pad them to the **max length in the batch**.
     - Use the index of `<pad>` for padding.
     - Return two tensors:
       - `src_batch` (shape: batch_size × max_src_len)
       - `tgt_batch` (shape: batch_size × max_tgt_len)




In [2]:
# 4.2 – Data: vocabulary, encoding, reversed targets, dataset

# TODO: build vocabulary from `corpus` can grab it from previous PL notebook
# - collect unique tokens
# - prepend ["<pad>", "<bos>", "<eos>"]
# - create `itos` (index to string) and `stoi` (string to index)
corpus = [
    "the cat sat on the mat",
    "the dog lay on the rug",
    "dogs and cats are friends",
    "the cat chased the mouse",
    "the dog chased the cat",
    "cats and dogs can be friends",
    "dogs are friends",
    "the dog sat on the mat"
]

tokens = []     
for sentence in corpus:
    tokens.extend(sentence.split())
special = ["<pad>", "<bos>", "<eos>"]
vocab = special + sorted(set(tokens))
stoi = {word: i for i, word in enumerate(vocab)}
itos = {i: word for word, i in stoi.items()}
vocab_size = len(vocab)

pad_idx = stoi["<pad>"]
bos_idx = stoi["<bos>"]
eos_idx = stoi["<eos>"]

print("Vocab:", vocab)
print("Vocab size:", vocab_size)

# TODO: indexed_corpus: list of lists of indices
indexed_corpus = [[stoi[word] for word in sentence.split()] for sentence in corpus]
print("indexed_corpus", indexed_corpus)

# TODO: retrieve pad, bos, eos indices from stoi


def encode_sentence(sentence: str):
    """
    TODO:
      - split sentence into tokens
      - map each token to its index
      - wrap with <bos> at start and <eos> at end
      - return list of ints
    """
    ids = [bos_idx] + [stoi[token] for token in sentence.split()] + [eos_idx]
    return ids


def encode_reversed(sentence: str):
    """
    TODO:
      - split sentence into tokens
      - reverse token order
      - map each reversed token to its index
      - wrap with <bos> and <eos>
      - return list of ints
    """
    ids = [bos_idx] + [stoi[token] for token in reversed(sentence.split())] + [eos_idx]
    return ids


class ReversalDataset(Dataset):
    """
    Dataset of (src_ids, tgt_ids) pairs for sentence reversal.
    src: original sentence with BOS/EOS
    tgt: reversed sentence with BOS/EOS
    """
    def __init__(self, corpus):
        # TODO:
        # - for each sentence in corpus
        #   - compute src_ids = encode_sentence(...)
        #   - compute tgt_ids = encode_reversed(...)
        #   - store pairs in a list
        self.pairs = [(encode_sentence(sentence), encode_reversed(sentence)) for sentence in corpus]

    def __len__(self):
        # TODO: return number of pairs

        return len(self.pairs)

    def __getitem__(self, idx):
        # TODO: return (src_ids, tgt_ids) for index idx
        return self.pairs[idx]

def reversal_collate_fn(batch):
    """
    Collate function for DataLoader.
    Input: list of (src_ids, tgt_ids)
    Output:
      - src_batch: LongTensor (B, max_src_len)
      - tgt_batch: LongTensor (B, max_tgt_len)
    Both padded with pad_idx.
    """
    # TODO:
    # - separate src and tgt sequences
    # - compute max_src_len and max_tgt_len
    # - allocate padded tensors filled with pad_idx
    # - copy each sequence into its row

    src_seqs, tgt_seqs = zip(*batch)
    max_src_len = max(len(seq) for seq in src_seqs)
    max_tgt_len = max(len(seq) for seq in tgt_seqs)
    batch_size = len(batch)

    src_batch = torch.full((batch_size, max_src_len), pad_idx, dtype=torch.long) 
    
    for i, seq in enumerate(src_seqs):
        src_batch[i, :len(seq)] = torch.tensor(seq, dtype=torch.long)


    
    tgt_batch = torch.full((batch_size, max_tgt_len), pad_idx, dtype=torch.long) 
    for i, seq in enumerate(tgt_seqs):
        tgt_batch[i, :len(seq)] = torch.tensor(seq, dtype=torch.long)


    return src_batch, tgt_batch


# Instantiate dataset and dataloader (once the TODOs are implemented)
dataset = ReversalDataset(corpus)
loader = DataLoader(dataset, batch_size=4, shuffle=True, collate_fn=reversal_collate_fn)


Vocab: ['<pad>', '<bos>', '<eos>', 'and', 'are', 'be', 'can', 'cat', 'cats', 'chased', 'dog', 'dogs', 'friends', 'lay', 'mat', 'mouse', 'on', 'rug', 'sat', 'the']
Vocab size: 20
indexed_corpus [[19, 7, 18, 16, 19, 14], [19, 10, 13, 16, 19, 17], [11, 3, 8, 4, 12], [19, 7, 9, 19, 15], [19, 10, 9, 19, 7], [8, 3, 11, 6, 5, 12], [11, 4, 12], [19, 10, 18, 16, 19, 14]]


## 4.3 — Positional Encoding

Transformers do not have recurrence (like RNNs) or convolution (like CNNs), so
they **cannot know the order of the sequence by themselves**.
We must explicitly add information about each token’s **position**.

We will implement the standard **sinusoidal positional encoding** used in the
original Transformer paper. Check Theoretical slides for more.

### Concept

- Each position `pos` in the sequence receives a vector of size `d_model`.
- This vector has a **pattern of sine and cosine waves** with different
  frequencies.
- These patterns allow the model to:
  - know *which* token is earlier/later,
  - measure relative distances,
  - generalize to longer sequences not seen during training.

### Why sin/cos?

- Sine and cosine are continuous and periodic.
- Different frequencies encode different scales:
  - Some dimensions change slowly → capture long-range structure.
  - Others change quickly → capture local order.

This means the model can infer relations like  
“token 3 is close to token 4, but far from token 10”.

###  What you need to implement

Your task is to:

1. Create a matrix `pe` of shape `(max_len, d_model)`.
2. For each position `pos`:
   - The **even dimensions** use a sine function.
   - The **odd dimensions** use a cosine function.
3. Register the positional encoding so it is not trained.
4. In the `forward` pass:
   - Add the positional encoding to the input embedding.
   - Apply dropout.

### Implementation logic (without code)

Think of it like building a table:

| position | dim 0 | dim 1 | dim 2 | dim 3 | ... |
|----------|--------|--------|--------|--------|-----|
| 0        | sin(0) | cos(0) | sin(0 / big) | cos(0 / big) | ... |
| 1        | sin(1) | cos(1) | sin(1 / big) | cos(1 / big) | ... |
| 2        | sin(2) | cos(2) | sin(2 / big) | cos(2 / big) | ... |

Where `big` is a large number controlling how fast each dimension changes.

- Even dims: **sin(pos / frequency)**
- Odd dims: **cos(pos / frequency)**  
- Different dims use different `frequency` values.

### Small example (d_model = 4)

Imagine you want positional encodings for positions 0, 1, 2:
pos = 0 → [sin(0), cos(0), sin(0 / K), cos(0 / K)]
pos = 1 → [sin(1), cos(1), sin(1 / K), cos(1 / K)]
pos = 2 → [sin(2), cos(2), sin(2 / K), cos(2 / K)]

Notice:
- sin(0) = 0
- cos(0) = 1  
So row 0 always starts with `[0, 1, 0, 1]`.

### How it attaches to embeddings

Every token embedding `E` of shape `(B, T, d_model)` becomes:

E_with_pos = E + pe[:T]

This lets the Transformer know the order.

---

Now complete the next code cell:

- Fill in the constructor to build the `pe` matrix.
- Add the positional embeddings in the `forward()` method.

In [3]:
# 4.3 – Positional Encoding 

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

        position = torch.arange(0, max_len).float().unsqueeze(1)  # (max_len, 1)
        encoding_matrix = torch.zeros(max_len, d_model)

        for i in range(0, d_model, 2):
            big = 10000 ** (i / d_model)
            encoding_matrix[:, i] = torch.sin(position.squeeze() / big)

        for i in range(1, d_model, 2):
            big = 10000 ** ((i - 1) / d_model)
            encoding_matrix[:, i] = torch.cos(position.squeeze() / big)

        self.register_buffer("pe", encoding_matrix)  # TODO: replace None with the encoding matrix

    def forward(self, x):
        # x: (batch, time, d_model)
        # TODO: add positional encoding to the input and apply dropout

        x = x + self.pe[: x.size(1), :].unsqueeze(0)
        x = self.dropout(x)
        return x


## 4.4 — Masks: Padding Mask and Causal Mask

Transformers rely heavily on **attention**, meaning each token may “look at”
every other token.  
But for training a sequence-to-sequence model (like reversal), two important
rules must be enforced:

1. **Padding should not be attended to**  
2. **The decoder should not look into the future**

We accomplish this with **masks**.

---

## 1. Padding Mask (for encoder & decoder)

When we batch sequences of different lengths, we pad them:

[False, False, False, True, True]


Later expanded to a shape that fits the attention mechanism.

### Concept

- Input: tensor of shape `(B, T)` with token indices.
- Output: mask of shape `(B, 1, 1, T)` with:
  - `True` in positions that are padding
  - `False` everywhere else

This ensures the model ignores padded tokens.

---

## 2. Causal Mask (decoder self-attention)

When the decoder is predicting token *i*, it **must not look at tokens > i**.

Example with sequence length T = 4:

### Valid (allowed)
A token can attend to:
- itself
- previous tokens

### Invalid (disallowed)
A token must NOT attend to:
- future tokens

We construct a **(T, T)** matrix like:

[False, False, False, True, True]

Later expanded to a shape that fits the attention mechanism.

### Concept

- Input: tensor of shape `(B, T)` with token indices.
- Output: mask of shape `(B, 1, 1, T)` with:
  - `True` in positions that are padding
  - `False` everywhere else

This ensures the model ignores padded tokens.

---

## 2. Causal Mask (decoder self-attention)

When the decoder is predicting token *i*, it **must not look at tokens > i**.

Example with sequence length T = 4:

### Valid (allowed)
A token can attend to:
- itself
- previous tokens

### Invalid (disallowed)
A token must NOT attend to:
- future tokens

We construct a **(T, T)** matrix like:

$
\begin{array}{c|cccc}
          & 0 & 1 & 2 & 3 \\ \hline
\text{token }0 & 0 & -\infty & -\infty & -\infty \\
\text{token }1 & 0 & 0       & -\infty & -\infty \\
\text{token }2 & 0 & 0       & 0       & -\infty \\
\text{token }3 & 0 & 0       & 0       & 0
\end{array}
$


Where:
- `0` means "allowed"
- `-∞` means "block this attention"

### Why this matters?

This prevents the model from cheating:
- At time step 3, the decoder should not know what token 4 will be.

---

## What you need to implement (summary)

### Padding Mask
- Input: `(B, T)`
- Output: `(B, 1, 1, T)`
- Mark positions equal to the pad index as **True**

### Causal Mask
- Input: integer `T`
- Output: `(T, T)` matrix
- Set **future** positions to `-inf`
- Set current/past positions to `0`

### High-level pseudocode 
padding_mask(seq):
    # seq has shape (B, T)
    
    1. For each position, check if it equals the pad token index.
       This produces a boolean mask of shape (B, T):
           True  → padding
           False → real token
    
    2. Reshape the mask to shape (B, 1, 1, T)
       so it can be broadcast correctly inside multi-head attention.
    
    3. Return the padding mask.


causal_mask(T):
    # T is the target sequence length
    
    1. Create a T×T matrix initialized with zeros.
    
    2. For all positions where column > row (j > i),
       set the value to -∞ to block attention to future tokens.
       
       Example for T = 4:
           [[ 0,   -∞, -∞, -∞ ],
            [ 0,    0, -∞, -∞ ],
            [ 0,    0,  0, -∞ ],
            [ 0,    0,  0,  0 ]]
    
    3. Return the causal mask.



In [4]:
# 4.4 – Masks 

def make_padding_mask(seq, pad_idx):
    """
    seq: (batch, time)
    returns: (batch, 1, 1, time) mask with True at padding positions.

    Create a boolean mask where seq == pad_idx and reshape it to
    (B, 1, 1, T) so it can be broadcast correctly inside multi-head attention.
    """

    # seq is (B, T) -> (B, T) boolean mask
    mask = (seq == pad_idx)

    # reshape to (B, 1, 1, T)
    return mask.unsqueeze(1).unsqueeze(1)


def generate_causal_mask(size):
    """
    Create a (size, size) causal mask for decoder self-attention.

    Returns a float tensor where positions above the diagonal are -inf
    (to block attention) and on/under the diagonal are 0.
    """

    # Create a float matrix with zeros and set -inf above diagonal
    matrix = torch.zeros(size, size, dtype=torch.float32)
    for i in range(size):
        for j in range(size):
            if j > i:
                matrix[i, j] = float('-inf')
    return matrix


## 4.5 — Building the TransformerSeq2Seq Model

In this section, we build the neural architecture that will learn to reverse
sentences using the Transformer encoder–decoder.

PyTorch already provides the core building block (`nn.Transformer`), so our job
is to construct the **input/output layers** around it and feed everything
into the Transformer in the right shape.

---

## Components you need to implement

A complete Transformer sequence-to-sequence model has:

### 1. **Token Embeddings**
Each token index is mapped to a vector of size `d_model`.  
We need two embedding tables:
- `src_emb` for source sentences
- `tgt_emb` for target sentences

These are learned during training.

---

### 2. **Positional Encoding**
Transformers require positional information to understand ordering.

You will:
- Pass the token embeddings through the `PositionalEncoding` module you wrote.
- Apply it to **both** source and target embeddings.

---

### 3. **The Transformer Module**
We use:

nn.Transformer(
d_model,
nhead,
num_encoder_layers,
num_decoder_layers,
dim_feedforward,
dropout,
batch_first = False
)


Important shapes for `nn.Transformer`:
- Input must be `(sequence_length, batch, d_model)`
- Output is `(sequence_length, batch, d_model)`

So you must **transpose**:
- from `(B, S, E)` → `(S, B, E)`  
- from `(B, T, E)` → `(T, B, E)`  
before feeding them to the transformer.

And transpose the result back.

---

### 4. **Projection to Vocabulary**
The decoder produces one vector of size `d_model` per time step.  
We convert this into vocabulary-sized logits using:

nn.Linear(d_model, vocab_size)


This is how we predict the next token.

---

## 🔁 Forward Pass Logic (conceptual)

You will implement this logic inside `forward(src, tgt_in)`:

### **Step 1 — Embed src and tgt**
-- src_emb = src_emb(src) # (B, S, E)
-- tgt_emb = tgt_emb(tgt_in) # (B, T, E)


### **Step 2 — Scale embeddings**
This helps stabilize training:
-- emb *= sqrt(d_model)


### **Step 3 — Add positional encodings**
-- src_pe = PE(src_emb)
-- tgt_pe = PE(tgt_emb)


### **Step 4 — Build masks**
You will use the functions from Part 4.4:
- padding mask for src
- padding mask for tgt_in
- causal mask for the decoder

### **Step 5 — Transpose for Transformer**
Transformers expect:
(S, B, E) and (T, B, E)


### **Step 6 — Call the Transformer**
The PyTorch transformer internally handles:
- multi-head attention
- residual connections
- feedforward layers
- layer normalization

You only supply:
- encoder input
- decoder input
- masks

### **Step 7 — Project outputs to vocabulary**
-- logits = proj(out)


This gives the final shape:
(B, T, vocab_size)


---

### Very small example (for intuition)

Check the following example:
```
src = ["the", "cat"]
tgt_in = ["<bos>", "cat", "the"]
```

Steps:

tokens → embeddings → + position → transformer → logits → next tokens


The model learns the mapping:

["the", "cat"] → ["cat", "the"]


But generalized to any sentence in the corpus.

---

###  Your tasks in the next code cell

Inside the class:

- Initialize `src_emb`, `tgt_emb`
- Create a projection layer
- Add a `PositionalEncoding` instance
- Build the `nn.Transformer`
- In `forward()`:
  - Apply embeddings
  - Add positional encodings
  - Build masks
  - Transpose shapes
  - Call the transformer
  - Project the result to vocabulary size

Only the **concept** is given here — your implementation must fill in the details.



In [5]:
# 4.5 – TransformerSeq2Seq Model (Student)

class TransformerSeq2Seq(nn.Module):
    def __init__(
        self,
        vocab_size,
        d_model=128,
        nhead=4,
        num_layers=2,
        dim_ff=256,
        dropout=0.1,
        pad_idx=0
    ):
        super().__init__()

        # TODO: create src and tgt embeddings
        self.src_emb = nn.Embedding(vocab_size, d_model)
        self.tgt_emb = nn.Embedding(vocab_size, d_model)

        # TODO: positional encoding
        self.pe = PositionalEncoding(d_model, dropout=dropout)

        # TODO: transformer encoder–decoder
        self.tf = nn.Transformer(d_model=d_model, nhead=nhead, num_encoder_layers=num_layers, num_decoder_layers=num_layers, dim_feedforward=dim_ff, dropout=dropout, batch_first=False)

        # TODO: projection layer to vocabulary
        self.proj = nn.Linear(d_model, vocab_size)

        self.pad_idx = pad_idx
        self.d_model = d_model

    def forward(self, src, tgt_in):
        """
        src:   (B, S)
        tgt_in: (B, T)
        returns logits: (B, T, vocab_size)

        TODO:
          - embed src and tgt
          - scale embeddings
          - add positional encodings
          - build padding masks for src and tgt
          - build causal mask for tgt
          - switch to (S, B, E) and (T, B, E)
          - call transformer
          - transpose back
          - project to vocabulary size
        """
        # Embeddings
        src_emb = self.src_emb(src)
        tgt_emb = self.tgt_emb(tgt_in)

        # Scale
        src_emb_scaled = src_emb * math.sqrt(self.d_model)
        tgt_emb_scaled = tgt_emb * math.sqrt(self.d_model)

        # Positional encoding
        src_pe = self.pe(src_emb_scaled)
        tgt_pe = self.pe(tgt_emb_scaled)

        # Build padding masks (as boolean masks expected by nn.Transformer)
        # src_key_padding_mask: (B, S)
        src_key_padding_mask = (src == self.pad_idx)
        # tgt_key_padding_mask: (B, T)
        tgt_key_padding_mask = (tgt_in == self.pad_idx)

        # Causal mask for decoder self-attention (T, T) float mask with -inf
        causal_mask = generate_causal_mask(tgt_in.size(1)).to(src.device)

        # Transpose to (S, B, E) and (T, B, E)
        src_pe = src_pe.transpose(0, 1) # (S, B, E)
        tgt_pe = tgt_pe.transpose(0, 1) # (T, B, E)

        # Call transformer
        output = self.tf(
            src_pe,
            tgt_pe,
            src_key_padding_mask=src_key_padding_mask,
            tgt_key_padding_mask=tgt_key_padding_mask,
            memory_key_padding_mask=src_key_padding_mask,
            tgt_mask=causal_mask,
        )

        # Back to (B, T, E)
        output = output.transpose(0, 1) # (B, T, E)

        # Project to vocabulary
        logits = self.proj(output) # (B, T, vocab_size)
        return logits


## 4.6 — Training the Transformer

The goal of this block is to train our Transformer encoder–decoder model on the
sentence reversal task.

During training, the decoder should learn to predict the **next token** in the
reversed target sequence.  
To help it, we use **teacher forcing**: we feed the *correct* previous token
to the decoder, not its own past predictions.

---

## Training Mechanics

For each batch of `(src, tgt)`:

### 1. Prepare the decoder input (`tgt_in`)
`tgt` contains the full reversed target sequence with `<bos>` and `<eos>`.

We need a shifted version:

tgt = [<bos>, w1, w2, w3, <eos>]
tgt_in = [<bos>, <bos>, w1, w2, w3]


This ensures:
- at position `t`, the decoder sees the *correct* previous token,
- and must predict the next one.

The function `shift_right(tgt)` will do this.

---

## 2. Forward pass
Call the model:

logits = model(src, tgt_in)

Output shape:
- `(batch, target_length, vocab_size)`

---

## 3. Compute the loss
We use cross-entropy over vocabulary predictions:

loss = CrossEntropyLoss(ignore_index=pad_idx)


Ignoring padding lets us train only on meaningful positions.

The loss compares:
- `logits` (model predictions)
- `tgt`    (correct reversed sequence)

---

## 4. Backward pass
The training loop will:
- zero gradients
- backpropagate the loss
- optionally clip gradients (helps stability)
- update parameters with AdamW

---

## 5. Logging
We track:
- total loss
- loss per token

This helps see whether the model is converging.

---

## Small conceptual example

Suppose a target sequence is:

[BOS, "mat", "the", "on", EOS]


Then:

tgt_in = [BOS, BOS, "mat", "the"]
tgt_out = [BOS, "mat", "the", "on"] # (the model predicts this)


At each position the model predicts the next token in the reversed sentence.

---

## Your tasks in the next code cell:

- Implement `shift_right(tgt)`
- Implement the full training loop:
  - call the model
  - compute loss
  - update parameters
  - accumulate statistics

Be careful with:
- shapes of inputs,
- ignoring `<pad>` in the loss,
- calling `.to(device)` for tensors,
- switching model to `train()` mode.



In [ ]:
# 4.6 – Training the Transformer (Student)

def shift_right(tgt):
    """
    Input:  tgt (B, T) with BOS at t=0
    Output: shifted tgt_in (B, T)
    Shift right for teacher forcing: decoder input has BOS at first position,
    followed by all target tokens except final EOS.
    """
    tgt_in = torch.zeros_like(tgt)
    tgt_in[:, 1:] = tgt[:, :-1]  # move all tokens one step right
    tgt_in[:, 0] = bos_idx       # ensure BOS at position 0
    return tgt_in


def train_transformer(model, loader, epochs=10, lr=2e-3, clip=1.0):
    """
    Training loop for the Transformer model.

    Adjusted so that loss excludes the initial BOS token (predict real tokens).
    """
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss(ignore_index=model.pad_idx)

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0.0
        for src, tgt in loader:
            src = src.to(device)
            tgt = tgt.to(device)

            tgt_in = shift_right(tgt)

            logits = model(src, tgt_in)  # (B, T, vocab_size)

            # Exclude position 0 (BOS) from loss: compare predictions from t>=1
            loss = criterion(logits[:, 1:].reshape(-1, logits.size(-1)), tgt[:, 1:].reshape(-1))

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
            optimizer.step()

            epoch_loss += loss.item()

        avg_loss = epoch_loss / len(loader)
        print(f"Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}")

    # Return model for convenience
    return model


## 4.7 — Greedy Decoding (Inference Step)

Once the model has been trained, we want to *use it* to produce reversed
sentences from new inputs.

During inference (testing), we do **not** use teacher forcing.
Instead, we generate one token at a time:

1. Feed the source sentence (`src`) to the encoder.
2. Start the decoder with only `<bos>`.
3. Predict the next token.
4. Append it to the decoder input.
5. Predict again.
6. Repeat until:
   - the model outputs `<eos>`, or
   - we hit a maximum length limit.

This is called **greedy decoding**, because at each step we take the token with
the highest probability (argmax).

---

## Key idea

At time step *t*, the decoder receives:
- the full encoded source representation
- all previously predicted target tokens (from steps 1 to t–1)

Then it predicts token *t*.

This process builds the reversed sentence step by step.

---

## Example (conceptual)

Suppose the model is reversing:

src = ["the", "cat", "sat"]


Greedy decoding proceeds like this:

Step | Decoder Input | Model Predicts | Decoder Output
-----|----------------|----------------|----------------
0    | [BOS]          | "sat"          | [BOS, sat]
1    | [BOS, sat]     | "cat"          | [BOS, sat, cat]
2    | [BOS, sat, cat]| "the"          | [BOS, sat, cat, the]
3    | ...            | EOS            | Stop

Final result:

[sat, cat, the]

---

## What you need to implement

Your decoding function should:

1. Receive `src` as a list of token IDs.
2. Convert it to a batch of size 1.
3. Initialize the target input with `<bos>`.
4. In a loop:
   - Run the model on `(src, tgt_current)`
   - Read the last-step logits
   - `argmax` to pick the next token
   - Append it to `tgt_current`
   - Break if `<eos>` is produced
5. Return the full predicted sequence of IDs.

### Important:

- Use `model.eval()` so dropout is disabled.
- Wrap decoding in `torch.no_grad()` (no gradients needed).
- Always keep tensor shapes correct:
  - src: `(1, S)`
  - tgt: `(1, T_current)`

---

##  Output format

The function should return a list of integer token IDs.  
You can convert them into words later using the vocabulary mapping (`itos`).

Ready to implement it in the code cell below.

In [ ]:
# 4.7 – Greedy Decoding

@torch.no_grad()
def greedy_decode(model, src, max_len=40, suppress_bos=True):
    """
    Greedy decoding for Transformer seq2seq.

    suppress_bos: if True, after the first step the BOS token is prevented
    from being selected again (its logit set to -inf).
    """
    model.eval()

    # Ensure src is a list of ints
    if isinstance(src, torch.Tensor):
        src_list = src.tolist()
    else:
        src_list = src

    src_tensor = torch.tensor(src_list, dtype=torch.long).unsqueeze(0).to(device)  # (1, S)
    tgt = torch.tensor([bos_idx], dtype=torch.long).unsqueeze(0).to(device)        # (1, 1)

    for step in range(max_len - 1):
        logits = model(src_tensor, tgt)  # (1, T, vocab_size)
        next_token_logits = logits[:, -1, :]  # (1, vocab_size)

        if suppress_bos and step >= 0:  # suppress BOS after first prediction attempt
            next_token_logits[0, bos_idx] = -1e9

        next_token_id = torch.argmax(next_token_logits, dim=-1).unsqueeze(0)  # (1, 1)
        tgt = torch.cat([tgt, next_token_id], dim=1)

        if next_token_id.item() == eos_idx:
            break

    return tgt.squeeze(0).tolist()


## 4.8 — Putting It All Together: Train & Test

In this final part, you will:

1. **Instantiate** the `TransformerSeq2Seq` model with:
   - the vocabulary size from Part 4.2,
   - a reasonable `d_model` and number of layers.

2. **Train** the model using:
   - the `train_transformer(...)` function from Part 4.6,
   - the `DataLoader` built from the corpus.

3. **Test** the model qualitatively:
   - pick a few sentences from the corpus,
   - encode them with `encode_sentence(...)`,
   - use `greedy_decode(...)` to generate the reversed sequence,
   - decode token IDs back to words using the vocabulary mapping (`itos`).

The goal is **not** to get perfect performance, but to:

- Verify that your implementation of:
  - positional encoding,
  - masks,
  - Transformer forward pass,
  - training loop,
  - greedy decoding,

  is *consistent* and *runs end-to-end* without errors.

---

### ✅ Steps to follow

1. **Create the model**

Use `TransformerSeq2Seq(...)` with parameters like:

- `d_model`: 64 or 128
- `nhead`: 2 or 4
- `num_layers`: 2
- `dim_ff`: 128 or 256
- `pad_idx`: the padding index from your vocabulary

2. **Train**

Use a small number of epochs first (e.g. 5–10):

- Does the loss decrease?
- Does the model start to output meaningful reversed sequences?

3. **Test on a few sentences**

For each test sentence:

- Print:
  - the **original sentence**,
  - the **expected reversed sentence** (using Python string ops),
  - the **model prediction**.

Example output format:

```text
SRC:  the cat chased the mouse
GT: mouse the chased cat the
PRED: mouse the chased cat the

Even if it’s not perfect, you should see that:
- the model preserves the vocabulary,
- and often gets the order mostly correct after some epochs.

In [11]:
# 4.8 –  Train & Test - complete code but depends on finishing the above parts!

# Helper: convert ids back to a sentence (skipping special tokens)
def ids_to_sentence(ids, itos, bos_idx, eos_idx, pad_idx):
    words = []
    for idx in ids:
        if idx == bos_idx or idx == pad_idx:
            continue
        if idx == eos_idx:
            break
        words.append(itos[idx])
    return " ".join(words)

# Instantiate the model
model = TransformerSeq2Seq(
    vocab_size=vocab_size,
    d_model=128,
    nhead=4,
    num_layers=2,
    dim_ff=256,
    dropout=0.1,
    pad_idx=pad_idx
).to(device)

print("Training Transformer model on reversal task...")
train_transformer(model, loader, epochs=10, lr=2e-3, clip=1.0)

# Test on a few sentences from the corpus
test_sentences = [
    "the cat sat on the mat",
    "the dog chased the cat",
    "dogs and cats are friends",
    "the cat chased the mouse",
]

print("\n=== Qualitative evaluation ===\n")
for s in test_sentences:
    # Encode source
    src_ids = encode_sentence(s)
    src_tensor = torch.tensor(src_ids, dtype=torch.long)

    # Greedy decode
    pred_ids = greedy_decode(model, src_tensor, max_len=40)

    # Build "gold" reversed sentence via Python
    gold_words = s.split()[::-1]
    gold_str = " ".join(gold_words)

    # Decode predicted ids back to words
    pred_str = ids_to_sentence(pred_ids, itos, bos_idx, eos_idx, pad_idx)

    print(f"SRC : {s}")
    print(f"GT: {gold_str}")
    print(f"PRED: {pred_str}")
    print("-" * 50)

Training Transformer model on reversal task...
Epoch 1/10, Loss: 3.1109
Epoch 2/10, Loss: 2.3922
Epoch 3/10, Loss: 2.0670
Epoch 4/10, Loss: 1.7410
Epoch 5/10, Loss: 1.3726
Epoch 6/10, Loss: 1.1681
Epoch 7/10, Loss: 0.9379
Epoch 8/10, Loss: 0.7982
Epoch 9/10, Loss: 0.6858
Epoch 10/10, Loss: 0.6405

=== Qualitative evaluation ===

SRC : the cat sat on the mat
GT: mat the on sat cat the
PRED: mat the on sat cat the
--------------------------------------------------
Epoch 5/10, Loss: 1.3726
Epoch 6/10, Loss: 1.1681
Epoch 7/10, Loss: 0.9379
Epoch 8/10, Loss: 0.7982
Epoch 9/10, Loss: 0.6858
Epoch 10/10, Loss: 0.6405

=== Qualitative evaluation ===

SRC : the cat sat on the mat
GT: mat the on sat cat the
PRED: mat the on sat cat the
--------------------------------------------------


/tmp/ipykernel_54827/2184635726.py:23: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  src = torch.tensor(src).unsqueeze(0).to(device)  # Convert src to tensor shape (1, S)


SRC : the dog chased the cat
GT: cat the chased dog the
PRED: 
--------------------------------------------------
SRC : dogs and cats are friends
GT: friends are cats and dogs
PRED: friends are cats and dogs
--------------------------------------------------
SRC : the cat chased the mouse
GT: mouse the chased cat the
PRED: 
--------------------------------------------------
SRC : the cat chased the mouse
GT: mouse the chased cat the
PRED: 
--------------------------------------------------
